# Beijing air quality: three-class next-day AQI

This notebook trains a model to estimate tomorrow’s AQI class from today’s completed pollutant readings and weather summaries. Data are loaded from MySQL; the notebook does not read CSV files.

## Three AQI classes

- **0 — Clean:** AQI 0–100 (Excellent and Good)
- **1 — Light-to-moderate polluted:** AQI 101–200
- **2 — Heavily polluted:** AQI above 200

The shared HJ 633 labeling rules live in `air-quality-classification/app/aqi_labels.py`. This notebook uses those rules to build consistent AQI targets and trains XGBoost for next-day prediction.

## Before you run

1. Open the repository root folder `Data_Science` in Jupyter.
2. Select the Python interpreter or kernel from your project `.venv`.
3. Make sure MySQL is running and database `air_quality` has `aqi_daily`.
4. Run the notebook cells from the top, one at a time with **Shift+Enter**.

If the database tables are not prepared, run `air-quality-classification/notebooks/etl_data_processor.ipynb` and then `air-quality-classification/notebooks/etl.ipynb` first. Those setup notebooks require the original PRSA source files; this model notebook itself reads only MySQL.


## China AQI basis and six-level classification

This notebook uses China’s AQI breakpoints, not WHO guideline values. The Ministry response linked below explains the HJ 633-2012 rationale; the current technical regulation is HJ 633-2026.

The output has one row per station and calendar day. PM2.5, PM10, SO2, NO2, and CO use 24-hour daily means, requiring at least 20 valid hourly readings for each pollutant. O3 uses the day’s maximum trailing 8-hour average; each window requires 6 valid readings and a day requires at least 14 valid windows ending from 08:00 through 23:00. When the daily maximum 8-hour O3 concentration exceeds 800 µg/m³, the daily maximum 1-hour O3 concentration is used for its IAQI. CO source values are converted from µg/m³ to mg/m³.

Each pollutant IAQI is linearly interpolated from its concentration breakpoints. The overall daily AQI is the maximum available pollutant IAQI.

| Category code | AQI range | China category |
| ---: | ---: | --- |
| 1 | 0–50 | Excellent |
| 2 | 51–100 | Good |
| 3 | 101–150 | Lightly polluted |
| 4 | 151–200 | Moderately polluted |
| 5 | 201–300 | Heavily polluted |
| 6 | 301–500 | Severely polluted |

| AQI breakpoint | PM2.5 | PM10 | SO2 (24h) | NO2 (24h) | CO (24h) | O3 (8h) |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 50 | 30 | 50 | 50 | 40 | 2 | 100 |
| 100 | 60 | 120 | 150 | 80 | 4 | 160 |
| 150 | 115 | 250 | 475 | 180 | 14 | 215 |
| 200 | 150 | 350 | 800 | 280 | 24 | 265 |
| 300 | 250 | 420 | 1600 | 565 | 36 | 800 |
| 400 | 350 | 500 | 2100 | 750 | 48 | — |
| 500 | 500 | 600 | 2620 | 940 | 60 | — |

Concentration units are µg/m³ except CO, which is mg/m³. The input must contain no duplicate station-hour records.

Sources: [Ministry of Ecology and Environment, linked response](https://www.mee.gov.cn/xxgk2018/xxgk/xxgk13/201810/t20181015_662213_wh.html); [HJ 633-2026 standard page](https://www.mee.gov.cn/ywgz/fgbz/bz/bzwb/jcffbz/202602/t20260225_1144441.shtml).


## Evaluation plan

The first 80% of unique dates are used for training and the final 20% for the test. Four expanding time-series folds split training rows by date for model selection. Imputation and scaling happen inside each model pipeline, so they are fitted on training rows only.

The notebook uses XGBoost as the production model. Persistence (tomorrow equals today) and a training-majority predictor are retained only as simple evaluation baselines. Metrics include date-bootstrap 95% accuracy intervals, macro-F1, per-class precision and recall, confusion matrices, XGBoost accuracy by station, and accuracy within one class.


In [1]:
import ast
import json
import os
import sys
sys.dont_write_bytecode = True  # Keep Python imports from creating cache files in the source repository.
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, inspect, text
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from xgboost import XGBClassifier


In [2]:
# Find the repo root whether Jupyter started in the repo or in a child folder.
HERE = Path.cwd().resolve()
SEARCH = [HERE, *HERE.parents]
ROOT = next((p for p in SEARCH if (p / "air-quality-classification" / "app" / "aqi_labels.py").exists()), None)
if ROOT is None:
    ROOT = next((p for p in SEARCH if (p / "Notebooks" / "aqi_labels.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find the project root. Open the Data_Science folder containing air-quality-classification/ in Jupyter.")
APP_DIR = ROOT / "air-quality-classification" / "app"
if not APP_DIR.exists():
    APP_DIR = ROOT
OUT = ROOT / "air-quality-classification" / "data"
MODEL_DIR = ROOT / "air-quality-classification" / "models"
OUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(APP_DIR))
from aqi_labels import AQI_CATEGORIES, O3_1H_BREAKPOINTS, POLLUTANT_BREAKPOINTS, category_codes, interpolate_iaqi
print("Project root:", ROOT)
print("Shared AQI labeler:", Path(interpolate_iaqi.__code__.co_filename).resolve())


Project root: C:\Users\PLP\Desktop\DataScienceProjects\Data_Science
Shared AQI labeler: C:\Users\PLP\Desktop\DataScienceProjects\Data_Science\air-quality-classification\app\aqi_labels.py


In [3]:
STATIONS=["aotizhongxin","changping","dingling","dongsi","guanyuan","gucheng","huairou","nongzhanguan","shunyi","tiantan","wanliu","wanshouxigong"]
POLLUTANTS=["PM2.5","PM10","SO2","NO2","CO","O3"]
WEATHER=["TEMP_mean","TEMP_range","PRES_mean","DEWP_mean","RAIN_sum","WSPM_mean","wd_sin","wd_cos"]
FEATURES=POLLUTANTS+WEATHER
SOURCE_TABLE="aqi_daily"
LABELS={0:"Clean",1:"Light-to-moderate polluted",2:"Heavily polluted"}


In [4]:
def _normalize_interpolated(values):
    """Give aqi_labels the comma-separated representation written by etl.ipynb."""
    def one(v):
        if v is None or (isinstance(v,float) and np.isnan(v)): return "None"
        if isinstance(v,(list,tuple,set)): return ", ".join(str(x) for x in v) if v else "None"
        s=str(v).strip()
        if s.startswith(("[","(")):
            try:
                parsed=ast.literal_eval(s)
                if isinstance(parsed,(list,tuple,set)): return ", ".join(map(str,parsed)) if parsed else "None"
            except (ValueError,SyntaxError): pass
        return "None" if s in ("", "[]", "None", "nan") else s
    return values.map(one)


In [5]:
def load_daily():
    """Load daily station summaries directly from air_quality.aqi_daily."""
    url=os.environ.get("DB_URL", "mysql+pymysql://root:@localhost/air_quality")
    engine=create_engine(url,pool_pre_ping=True)
    try:
        with engine.connect() as con:
            frame=pd.read_sql_query(text(f"SELECT * FROM {SOURCE_TABLE}"),con)
    except Exception as exc:
        raise RuntimeError(f"Could not load air_quality.{SOURCE_TABLE} from MySQL: {exc}") from exc
    finally:
        engine.dispose()
    required=["date","station","station_id",*POLLUTANTS,*WEATHER,"O3_1h_max"]
    missing=[c for c in required if c not in frame]
    if missing: raise KeyError(f"air_quality.{SOURCE_TABLE} is missing columns: {missing}")
    if frame.duplicated(["station_id","date"]).any(): raise ValueError("aqi_daily contains duplicate station-days")
    print(f"Loaded {len(frame):,} daily rows from air_quality.{SOURCE_TABLE}")
    return frame,[SOURCE_TABLE]


In [6]:
def build_labels_from_daily(frame):
    """Calculate AQI and pollutant categories from aqi_daily summaries."""
    daily=frame.copy()
    daily["date"]=pd.to_datetime(daily["date"],errors="coerce").dt.normalize()
    iaqi_columns=[]
    for pollutant in POLLUTANTS:
        values=pd.to_numeric(daily[pollutant],errors="coerce")
        col=f"{pollutant}_iaqi"
        if pollutant=="O3":
            sub=interpolate_iaqi(values,POLLUTANT_BREAKPOINTS["O3"])
            fallback=interpolate_iaqi(pd.to_numeric(daily["O3_1h_max"],errors="coerce"),O3_1H_BREAKPOINTS)
            use=values.gt(800)
            sub.loc[use]=fallback.loc[use]
        else:
            sub=interpolate_iaqi(values,POLLUTANT_BREAKPOINTS[pollutant])
        daily[col]=sub.round()
        daily[f"{pollutant}_category"]=category_codes(daily[col]).map(AQI_CATEGORIES)
        iaqi_columns.append(col)
    daily["aqi"]=daily[iaqi_columns].max(axis=1,skipna=True)
    daily["aqi_category_code"]=category_codes(daily["aqi"])
    daily["aqi_category"]=daily["aqi_category_code"].map(AQI_CATEGORIES)
    daily["main_pollutant"]=daily[iaqi_columns].fillna(-1).idxmax(axis=1).str.removesuffix("_iaqi")
    daily.loc[daily[iaqi_columns].isna().all(axis=1),"main_pollutant"]=np.nan
    return daily

def three_class_dataset(source):
    daily=source.copy() if "aqi" in source.columns else build_labels_from_daily(source)
    counts=daily["aqi_category"].value_counts(dropna=True).sort_index().to_dict()
    future=daily[["station_id","date","aqi","main_pollutant"]].copy()
    future["date"]=future["date"]-pd.Timedelta(days=1)
    future=future.rename(columns={"aqi":"target_AQI","main_pollutant":"target_main_pollutant"})
    today=daily[["station_id","date","station",*FEATURES]].copy()
    data=today.merge(future,on=["station_id","date"],how="inner",validate="one_to_one")
    data=data.dropna(subset=["target_AQI"]).copy()
    data["target_AQI"]=pd.to_numeric(data["target_AQI"],errors="coerce")
    data=data.dropna(subset=["target_AQI"])
    data["target_class"]=np.select([data.target_AQI<=100,data.target_AQI<=200],[0,1],default=2).astype(int)
    data["date"]=pd.to_datetime(data["date"]).dt.normalize()
    data=data.sort_values(["date","station"]).reset_index(drop=True)
    print("Daily HJ 633 AQI category counts:",json.dumps({str(k):int(v) for k,v in counts.items()}))
    print(f"Next-day examples: {len(data):,}; dates: {data.date.nunique():,}")
    return data,counts


In [7]:
def _log_features(frame):
    x=np.asarray(frame,dtype=float).copy()
    # log1p for nonnegative pollutant concentrations, rainfall and wind speed only.
    for i in [0,1,2,3,4,5,10,11]: x[:,i]=np.log1p(np.maximum(x[:,i],0))
    return x


In [8]:
def make_models():
    xgb = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(
            objective="multi:softprob", num_class=3, eval_metric="mlogloss",
            random_state=42, n_jobs=1, verbosity=0,
        )),
    ])
    grid = {"model__n_estimators": [120], "model__max_depth": [3], "model__learning_rate": [0.06]}
    return {"XGBoost": (xgb, grid)}


## Connect to MySQL and build supervised examples

This cell checks `DB_URL` (or uses the default local MySQL connection), reads `aqi_daily`. It then calculates daily AQI labels and creates exact next-day targets. If this step reports a connection error, start MySQL and prepare the database tables before continuing.


In [9]:
daily_source, source_tables = load_daily()
daily_source = build_labels_from_daily(daily_source)
data, canonical_label_counts = three_class_dataset(daily_source)
data.head()


Loaded 51,920 daily rows from air_quality.aqi_daily
Daily HJ 633 AQI category counts: {"Excellent": 11624, "Good": 18239, "Heavily polluted": 2330, "Lightly polluted": 13081, "Moderately polluted": 4358, "Severely polluted": 646}
Next-day examples: 50,266; dates: 4,243


,station_id,date,station,PM2.5,PM10,SO2,NO2,CO,O3,TEMP_mean,TEMP_range,PRES_mean,DEWP_mean,RAIN_sum,WSPM_mean,wd_sin,wd_cos,target_AQI,target_main_pollutant,target_class
0,1011A,2013-03-01,Aotizhongxin,7.125000,10.750000,11.708333,22.583333,0.429167,74.500,0.508333,9.0,1026.658333,-17.933333,0.0,3.415000,-0.015858,0.874674,83.0,NO2,0
1,1010A,2013-03-01,Changping,5.083333,18.958333,16.041667,15.333333,0.387500,88.875,0.845833,8.2,1023.787500,-16.337500,0.0,3.524583,-0.484243,0.867750,75.0,NO2,0
2,1002A,2013-03-01,Dingling,6.375000,6.270833,3.000000,2.333333,0.204167,86.000,0.504167,8.2,1017.029167,-16.666667,0.0,3.524583,-0.484243,0.867750,70.0,NO2,0
3,1003A,2013-03-01,Dongsi,6.416667,9.875000,8.291667,28.229167,0.389583,88.000,0.508333,9.0,1026.516667,-17.933333,0.0,3.415000,-0.015858,0.874674,73.0,NO2,0
4,1006A,2013-03-01,Guanyuan,7.541667,11.666667,8.500000,28.500000,0.400000,74.000,1.270833,8.7,1025.958333,-16.920833,0.0,3.115000,-0.289323,0.863489,79.0,NO2,0


## Split dates and train the models

This is the chronological holdout. Do not shuffle these dates: keeping the final dates untouched is what makes the test a realistic future-period evaluation. The next cell tunes each model only on the training rows.


In [10]:
dates = np.array(sorted(data.date.unique()))
cut = int(len(dates) * 0.80)
train_dates, test_dates = dates[:cut], dates[cut:]
train = data[data.date.isin(train_dates)].copy()
test = data[data.date.isin(test_dates)].copy()
print("Train dates:", train.date.min().date(), "through", train.date.max().date(), "rows:", len(train))
print("Test dates:", test.date.min().date(), "through", test.date.max().date(), "rows:", len(test))


Train dates: 2013-03-01 through 2022-08-31 rows: 40266
Test dates: 2022-09-01 through 2025-01-03 rows: 10000


In [11]:
date_to_position = {date: position for position, date in enumerate(train_dates)}
train_date_positions = train.date.map(date_to_position).to_numpy()
folds = []
for left_dates, right_dates in TimeSeriesSplit(n_splits=4).split(train_dates):
    left_rows = np.flatnonzero(np.isin(train_date_positions, left_dates))
    right_rows = np.flatnonzero(np.isin(train_date_positions, right_dates))
    folds.append((left_rows, right_rows))
fitted, predictions, probabilities = {}, {}, {}
class_labels = np.sort(train.target_class.unique())
for name, (pipeline, grid) in make_models().items():
    search = GridSearchCV(pipeline, grid, scoring="f1_macro", cv=folds, n_jobs=1, refit=True)
    search.fit(train[FEATURES], train.target_class)
    fitted[name] = search.best_estimator_
    predictions[name] = search.predict(test[FEATURES])
    raw_probabilities = search.predict_proba(test[FEATURES])
    probabilities[name] = np.zeros((len(test), len(class_labels)))
    probabilities[name][:, np.searchsorted(class_labels, search.classes_)] = raw_probabilities
    print(name, "best CV macro-F1:", round(search.best_score_, 4), "parameters:", search.best_params_)


XGBoost best CV macro-F1: 0.5975 parameters: {'model__learning_rate': 0.06, 'model__max_depth': 3, 'model__n_estimators': 120}


## Baselines, metrics, and saved outputs

Persistence predicts the AQI class observed on the preceding calendar date for the same station. The majority baseline always predicts the most frequent training class. The final cell writes the baseline comparison and XGBoost model artifact into `data/` and `models/`.

Read `results.csv` first for model accuracy compared with baselines, then `per_class.csv` to see which classes are difficult. `per_station.csv` contains ensemble accuracy by station. The report template is `REPORT.md`; replace its unmeasured references with the results after this notebook finishes.


In [12]:
y = test.target_class.to_numpy()
dates_test = test.date.to_numpy()
today = daily_source[["station", "date", "aqi"]].copy()
today["date"] = pd.to_datetime(today["date"]).dt.normalize() + pd.Timedelta(days=1)
today_aqi = pd.to_numeric(today["aqi"], errors="coerce")
today["today_class"] = np.select([today_aqi.le(100), today_aqi.le(200)], [0, 1], default=2)
today = today[["station", "date", "today_class"]]
joined = test[["station", "date"]].merge(today, on=["station", "date"], how="left")
majority = int(train.target_class.value_counts().idxmax())
predictions["Persistence"] = joined.today_class.fillna(majority).astype(int).to_numpy()
predictions["Majority class"] = np.full(len(test), majority, dtype=int)
rng = np.random.default_rng(42)
unique_test_dates = np.unique(dates_test)
by_date = {date: np.flatnonzero(dates_test == date) for date in unique_test_dates}
results_rows, class_rows, confusion_rows = [], [], []
for name, pred in predictions.items():
    accuracy = accuracy_score(y, pred)
    boot = []
    for _ in range(1000):
        sample = rng.choice(unique_test_dates, size=len(unique_test_dates), replace=True)
        indices = np.concatenate([by_date[date] for date in sample])
        boot.append(accuracy_score(y[indices], pred[indices]))
    low, high = np.quantile(boot, [0.025, 0.975])
    precision, recall, f1, _ = precision_recall_fscore_support(y, pred, labels=[0, 1, 2], zero_division=0)
    results_rows.append({"model": name, "accuracy": accuracy, "accuracy_ci95_low": low, "accuracy_ci95_high": high,
                         "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
                         "accuracy_within_one_class": float(np.mean(np.abs(y - pred) <= 1))})
    for klass in range(3):
        class_rows.append({"model": name, "class": klass, "class_label": LABELS[klass], "precision": precision[klass],
                           "recall": recall[klass], "f1": f1[klass], "support": int((y == klass).sum())})
    matrix = confusion_matrix(y, pred, labels=[0, 1, 2])
    for actual in range(3):
        for predicted in range(3):
            confusion_rows.append({"model": name, "actual_class": actual, "actual_label": LABELS[actual],
                                   "predicted_class": predicted, "predicted_label": LABELS[predicted],
                                   "count": int(matrix[actual, predicted])})
results = pd.DataFrame(results_rows)
per_class = pd.DataFrame(class_rows)
confusion = pd.DataFrame(confusion_rows)
test_with_predictions = test[["station", "target_class"]].copy()
test_with_predictions["predicted_class"] = predictions["XGBoost"]
per_station = pd.DataFrame([
    {"station": station, "rows": len(group), "accuracy": accuracy_score(group.target_class, group.predicted_class)}
    for station, group in test_with_predictions.groupby("station")
])
results.to_csv(OUT / "results.csv", index=False)
per_class.to_csv(OUT / "per_class.csv", index=False)
confusion.to_csv(OUT / "confusion_matrix.csv", index=False)
per_station.to_csv(OUT / "per_station.csv", index=False)
model_name = "XGBoost"
final_model = clone(fitted[model_name])
final_model.fit(data[FEATURES], data.target_class)
selected_metrics = results.loc[results.model == model_name].iloc[0].to_dict()
artifact = {"model": final_model, "model_name": model_name, "feature_order": FEATURES, "class_labels": LABELS,
            "training_period": [str(data.date.min().date()), str(data.date.max().date())],
            "test_period": [str(test.date.min().date()), str(test.date.max().date())], "metrics": selected_metrics,
            "source_tables": source_tables, "canonical_label_counts": {str(k): int(v) for k, v in canonical_label_counts.items()},
            "training_rows": len(data)}
joblib.dump(artifact, MODEL_DIR / "3class_model.joblib")
print(results.to_string(index=False))
print("Selected model:", model_name)
print("Canonical source AQI category counts:", canonical_label_counts)
results


         model  accuracy  accuracy_ci95_low  accuracy_ci95_high  macro_f1  accuracy_within_one_class
       XGBoost    0.7814           0.760145            0.802603  0.518231                     0.9956
   Persistence    0.6269           0.600818            0.654853  0.403887                     0.9735
Majority class    0.7104           0.683962            0.736653  0.276894                     0.9859
Selected model: XGBoost
Canonical source AQI category counts: {'Excellent': 11624, 'Good': 18239, 'Heavily polluted': 2330, 'Lightly polluted': 13081, 'Moderately polluted': 4358, 'Severely polluted': 646}


,model,accuracy,accuracy_ci95_low,accuracy_ci95_high,macro_f1,accuracy_within_one_class
0,XGBoost,0.7814,0.760145,0.802603,0.518231,0.9956
1,Persistence,0.6269,0.600818,0.654853,0.403887,0.9735
2,Majority class,0.7104,0.683962,0.736653,0.276894,0.9859
